# Análise de Velocidade de Leitura (VELmed) — Projeto SESI
## LMM + Visualização por Grupo e Sessão

## Célula 1 — Importações

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas.")

## Célula 2 — Configurações
> **Edite aqui** os caminhos, legendas, cores e sessões antes de rodar.

In [ ]:
# ── Caminho dos dados ──────────────────────────────────────────────────────
CAMINHO_DADOS = r'C:\Users\lucas\OneDrive\Documentos\Documents\1.DOUTORADO\SESI_01\PROC_DATA\aceletra_g_acelerado\sessao_stats.xlsx'

# ── Sessões a analisar ─────────────────────────────────────────────────────
SESSOES = list(range(0, 10))   # sessões 0 a 9

# ── Legendas do gráfico (edite livremente) ─────────────────────────────────
TITULO_GRAFICO  = 'Evolução do Tempo de Leitura por Grupo'
EIXO_X          = 'Sessão'
EIXO_Y          = 'Tempo médio de leitura (ms/caractere)'
LEGENDA_GRUPO_0 = 'Não acelerado'
LEGENDA_GRUPO_1 = 'Acelerado'
COR_GRUPO_0     = '#2166AC'
COR_GRUPO_1     = '#D6604D'
TAMANHO_FONTE   = 13

# ── Controles de exibição ──────────────────────────────────────────────────
MOSTRAR_PONTOS_ORIGINAIS = True   # False para ocultar scatter dos dados brutos
MOSTRAR_ANOTACAO_LMM     = True   # False para ocultar anotação LMM no gráfico
ALPHA_PONTOS             = 0.2    # transparência dos pontos (0.0 a 1.0)

# ── Escala do eixo Y (None = automático) ───────────────────────────────────
YLIM_MIN = None   # ex: 20
YLIM_MAX = None   # ex: 120

# ── Saídas ─────────────────────────────────────────────────────────────────
OUTPUT_EXCEL   = 'resultados_velmed.xlsx'
OUTPUT_GRAFICO = 'visualizacao_resultado_lmm.png'

print("Configurações definidas.")

## Célula 3 — Carregamento e formato longo

In [ ]:
df = pd.read_excel(CAMINHO_DADOS)
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print(f"Grupos: {df['grupo'].value_counts().to_dict()}")

# Formato longo
df_long = pd.melt(
    df,
    id_vars    = ['SUBJID', 'grupo'],
    value_vars = [f'VELmed{i}' for i in SESSOES],
    var_name   = 'session_raw',
    value_name = 'VELmed'
)
df_long['session'] = df_long['session_raw'].str.extract(r'(\d+)').astype(int)
df_long['grupo']   = df_long['grupo'].astype('category')

# Mapeamento de legendas
label_map = {0: LEGENDA_GRUPO_0, 1: LEGENDA_GRUPO_1}
df_long['grupo_label'] = df_long['grupo'].map(label_map)
palette = {LEGENDA_GRUPO_0: COR_GRUPO_0, LEGENDA_GRUPO_1: COR_GRUPO_1}

print(f"Formato longo: {df_long.shape}")
print(f"Sujeitos: {df_long['SUBJID'].nunique()}")
print(f"Sessões:  {sorted(df_long['session'].unique())}")
print(df_long.head(4))

## Célula 4 — Estatísticas descritivas

In [ ]:
resumo = (
    df_long
    .groupby(['session', 'grupo_label'])['VELmed']
    .agg(
        n       = 'count',
        media   = 'mean',
        dp      = 'std',
        epm     = 'sem',
        mediana = 'median',
        minimo  = 'min',
        maximo  = 'max',
    )
    .round(2)
    .reset_index()
)

print("=== Estatísticas descritivas por sessão e grupo ===")
print(resumo.to_string(index=False))

## Célula 5 — LMM (Modelo Linear Misto)
Modelo: `VELmed ~ sessão × grupo + (1|sujeito)`

In [ ]:
print("=" * 55)
print("  LMM — VELmed ~ sessão × grupo + (1|sujeito)")
print("=" * 55)

lmm_result = None

for metodo in ['bfgs', 'nm', 'lbfgs', 'powell']:
    try:
        lmm = smf.mixedlm(
            "VELmed ~ session * C(grupo)",
            data   = df_long,
            groups = df_long["SUBJID"]
        )
        lmm_result = lmm.fit(method=metodo, reml=True, disp=False)
        print(f"Método '{metodo}' convergiu.\n")
        break
    except Exception as e:
        print(f"Método '{metodo}' falhou: {type(e).__name__}")

if lmm_result is not None:
    print(lmm_result.summary())

    print("\n=== RESUMO DOS EFEITOS ===")
    params = lmm_result.params
    pvals  = lmm_result.pvalues
    for nome, beta, p in zip(params.index, params.values, pvals.values):
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
        print(f"  {nome:<52} β={beta:+.4f}  p={p:.4f}  {sig}")
else:
    print("Nenhum método convergiu.")

## Célula 6 — Previsões do modelo

In [ ]:
# Cria dataframe para previsões
predict_df = pd.DataFrame({
    'session': np.tile(np.array(SESSOES), 2),
    'grupo'  : np.repeat([0, 1], len(SESSOES))
})
predict_df['grupo'] = predict_df['grupo'].astype('category')
predict_df['grupo_label'] = predict_df['grupo'].map(label_map)

if lmm_result is not None:
    predict_df['predicted_VELmed'] = lmm_result.predict(predict_df)
    print("Previsões calculadas.")
    print(predict_df.head(6))
else:
    predict_df['predicted_VELmed'] = np.nan
    print("Modelo não disponível — previsões serão NaN.")

## Célula 7 — Gráfico
> Para abrir em **janela separada**: descomente `%matplotlib qt`  
> Para exibir **inline**: mantenha `%matplotlib inline` (padrão)

In [ ]:
# Descomente a linha desejada:
# %matplotlib qt       # janela separada (editável interativamente)
%matplotlib inline     

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('white')

# Pontos dos dados originais
if MOSTRAR_PONTOS_ORIGINAIS:
    for grupo_label, cor in palette.items():
        sub = df_long[df_long['grupo_label'] == grupo_label]
        ax.scatter(sub['session'], sub['VELmed'],
                   color=cor, alpha=ALPHA_PONTOS,
                   s=30, zorder=2)

# Linhas previstas pelo modelo
for grupo_label, cor in palette.items():
    sub = predict_df[predict_df['grupo_label'] == grupo_label].sort_values('session')
    ax.plot(sub['session'], sub['predicted_VELmed'],
            color=cor, linewidth=3,
            marker='o', markersize=7,
            markerfacecolor=cor,
            markeredgecolor='white', markeredgewidth=1,
            label=grupo_label, zorder=3)

# Anotação LMM
if MOSTRAR_ANOTACAO_LMM and lmm_result is not None:
    try:
        chaves = [k for k in lmm_result.params.index
                  if 'session' in k and 'grupo' in k.lower() and k != 'session']
        if chaves:
            k    = chaves[0]
            beta = lmm_result.params[k]
            p    = lmm_result.pvalues[k]
            sig  = '* p < 0.05' if p < 0.05 else 'n.s.'
            pstr = f'{p:.3f}' if p >= 0.001 else '< 0.001'
            ax.annotate(
                f'Sessão × Grupo: β = {beta:.4f}, p = {pstr} ({sig})',
                xy=(0.03, 0.05), xycoords='axes fraction',
                fontsize=10, color='#333333',
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='lightyellow',
                          edgecolor='grey', alpha=0.85)
            )
    except Exception:
        pass

# Escala Y
if YLIM_MIN is not None and YLIM_MAX is not None:
    ax.set_ylim(YLIM_MIN, YLIM_MAX)

# Formatação
ax.set_title(TITULO_GRAFICO,
             fontsize=TAMANHO_FONTE + 1, fontweight='bold', pad=14)
ax.set_xlabel(EIXO_X, fontsize=TAMANHO_FONTE)
ax.set_ylabel(EIXO_Y, fontsize=TAMANHO_FONTE)
ax.set_xticks(SESSOES)
ax.legend(title='Grupo', fontsize=11, title_fontsize=11,
          frameon=True, framealpha=0.9)
ax.grid(True, which='both', linestyle='--', linewidth=0.9, alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_GRAFICO, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Figura salva: {OUTPUT_GRAFICO}")
plt.show()

## Célula 8 — Exporta Excel

In [ ]:
# Tabela LMM
if lmm_result is not None:
    fe_idx = lmm_result.fe_params.index
    ci     = lmm_result.conf_int().loc[fe_idx]
    df_lmm = pd.DataFrame({
        'Preditor'  : fe_idx,
        'β'         : lmm_result.fe_params.values,
        'Std.Err.'  : lmm_result.bse.loc[fe_idx].values,
        'z'         : (lmm_result.fe_params /
                       lmm_result.bse.loc[fe_idx]).values,
        'p-valor'   : lmm_result.pvalues.loc[fe_idx].values,
        'IC 2.5%'   : ci[0].values,
        'IC 97.5%'  : ci[1].values,
    }).round(4)
else:
    df_lmm = pd.DataFrame({'Aviso': ['Modelo não convergiu']})

# Formato wide
df_wide = df[['SUBJID','grupo'] +
             [f'VELmed{s}' for s in SESSOES]].copy()
df_wide.columns = (['subject','grupo'] +
                   [f'VELmed_s{s}' for s in SESSOES])

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as w:
    df_lmm.to_excel(w,      sheet_name='LMM_VELmed',    index=False)
    resumo.to_excel(w,      sheet_name='Descritivas',    index=False)
    df_long.to_excel(w,     sheet_name='Formato_Longo',  index=False)
    df_wide.to_excel(w,     sheet_name='Formato_Wide',   index=False)
    predict_df.to_excel(w,  sheet_name='Previsoes_LMM',  index=False)

print(f"Excel salvo: {OUTPUT_EXCEL}")
print("\n=== ANÁLISE CONCLUÍDA ===")